# ***IMPORTS***

In [49]:
#pip install openpyxl
#pip install beautifulsoup4

import openpyxl

from bs4 import BeautifulSoup
import requests
import os
from datetime import datetime
import time
import winsound
import re
import urllib3
from urllib.parse import urljoin
#from colorama import init, Fore, Back, Style
#import pandas as pd

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


import warnings
warnings.filterwarnings('ignore')

## **FUNCIONES Y VARIABLES GLOBALES**

In [50]:
# Conectar y obtener datos desde Carrefour

#Se obtiene el User Agent desde http://httpbin.org/get
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36", "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}

def get_carrefour_price(sku):
    base_url = "https://www.carrefour.es/"
    query_url = base_url + f"?query={sku}"
    print(f"Solicitando: {query_url}")

    # 1. Petición inicial a la URL con el SKU
    response = requests.get(query_url, headers=headers, verify=False)
    soup = BeautifulSoup(response.content, "html.parser")

    # 2. Buscar redirección a la página del producto
    redirect_url = None
    for script in soup.find_all("script"):
        if script.string:
            match = re.search(r'window\.location\.href\s*=\s*"([^"]+)"', script.string)
            if match:
                redirect_url = match.group(1)
                break

    if not redirect_url:
        print("❌ No se encontró redirección a producto")
        return "No se pudo obtener el precio"

    # 3. Completar URL si es relativa
    redirect_url = urljoin(base_url, redirect_url)
    print(f"Redirigiendo manualmente a: {redirect_url}")

    # 4. Petición a la URL final del producto
    response = requests.get(redirect_url, headers=headers, verify=False)
    soup = BeautifulSoup(response.content, "html.parser")

    # 5. Buscar precio en el span con clase x-currency
    price_tag = soup.find("span", class_="x-currency")
    if price_tag:
        price = price_tag.text.strip()
        print(f"✅ Precio encontrado: {price}")
        return price
    else:
        print("❌ No se encontró el precio en la página del producto")
        return "No se pudo obtener el precio"


# #Funcion para obtener el precio de un producto
# def get_carrefour_price(web_prefix, ean):
#     url = web_prefix + ean
#     print(url)
#     #Se obtiene el HTML del producto
#     page = requests.get(url, headers=headers, verify=False)
#     time.sleep(3)
#     print(page.content.decode('utf-8')) 
#     soup1 = BeautifulSoup(page.content, "html.parser")

#     #Se extrae el precio del HTML
#     #try:
#     price = soup1.find("span", { "class" : "x-currency" }).text.strip() # type: ignore
#     print(price)
        
       
#     #except AttributeError:
#     #    price = "No se ha podido obtener el precio para este producto."
    
#     return price

#funciones para hora y fecha

def actual_date():
    now = datetime.now()
    date_now = now.date()
   
    return date_now

def actual_hour():
    now = datetime.now()
    hour_now = now.time()

    return hour_now

def pitido():
    frecuency = 1000
    duration = 100
    winsound.Beep(frecuency, duration)
    
def quit_two_firsts(string):
    return string[2:]


## **CONFIGURACIÓN DE ARCHIVO DE EXCEL**

In [ ]:
#Incluir la localizacion del fichero excel con los datos de carrefour:
directory = os.getcwd()
path = directory + "\\Precios.xlsx"
#print(directory)

# Se abre el workbook y la hoja seleccionados (donde se guardo el documento por ultima vez)
wb_obj = openpyxl.load_workbook(path)
sheet_obj = wb_obj.active

# Obtener el numero de filas y columnas usadas
row_limit = sheet_obj.max_row # type: ignore
column_limit = sheet_obj.max_column # type: ignore
  
print("Total de filas:", row_limit)
print("Total de columnas:", column_limit)

#Introducir el numero de columna en la que se encuentra el EAN (A=1, B=2, C=3...)
ean_col = 1

#Introducir el numero de la fila en la que se encuentra el primer valor de EAN
first_ean_row = 2

#Introducir el numero de columna en la que se encuentra el precio
precio_col = 3

## **EJECUTAR TAREA DE RELLENADO DE PRECIOS**

In [ ]:
# Cerrar el fichero Excel antes de ejecutar esta celda

#Introducir el formato de la web que se va a buscar
from alive_progress import alive_bar
from alive_progress import animations

web_prefix = "https://www.carrefour.es/?query="

date = actual_date()
hour = actual_hour()
time1 = time.time()

print(("INICIADO EL {} A LAS {} HORAS").format(date, hour))

for i in range(first_ean_row, row_limit + 1): 
    cell_obj = sheet_obj.cell(row = i, column = ean_col) # type: ignore
    ean1 = str(cell_obj.value)
    ean = quit_two_firsts(ean1)
    print(ean)
    time.sleep(3)
    if ean is None:
        ean = ""
    
    
    #url = web_prefix#+ean
    try:
        price = get_carrefour_price(web_prefix, ean)
        print(("{}: {} tiene un precio de: {}").format(str(i-1).zfill(4), ean, price))
    
        #Se introduce el precio en el Excel
        c1 = sheet_obj.cell(row = i, column = precio_col) # type: ignore
        c1.value = price
    except ConnectionError:
        #Manejo de caida de conexión
        bar = animations.bar_factory('😴', tip="😪", background='zZz', borders=('Durmiendo 👉 ->|','|<- Terminado 🤘'), errors=('<---👀', '💀'))
        print("Hemos tenido un error de conexión, voy a descansar 10 segundos. ZZZZZZzzzzz...")
        total = 10
        i = 0
        
        with alive_bar(total, tittle = "Durmiendo", bar=bar) as bar:
            while i < total:
                time.sleep(1)
                bar()
                i += 1
        
        time.sleep(10)
        print("Buena siesta!!! Vuelvo a la carga!!!!")
        try:
            price = get_carrefour_price(web_prefix, ean)
            print(("{}/{}: {} tiene un precio de: {}").format(str(i-1).zfill(4), (row_limit-1), ean, price))
    
            #Se introduce el precio en el Excel
            c1 = sheet_obj.cell(row = i, column = precio_col) # type: ignore
            c1.value = price
        except:
            break
    
#Se guarda el fichero Excel con los cambios aplicados
wb_obj.save(path)
time2 = time.time()
hour1 = actual_hour()
dif_time = time2 - time1
print(("FINALIZADO EL {} A LAS {} HORAS").format(date, hour1))
print(("REALIZADO EN: {} horas").format((dif_time/60)/60))
print("ARCHIVO GUARDADO / PROGRAMA FINALIZADO")
